In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import argparse
import csv

parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(parent_dir)

import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
import torchvision
import torchvision.transforms as transforms

from models.bayesian.resnet_variational import ResNet
from trainer import train_one_epoch, evaluate, train_one_epoch_bayesian, validate_bayesian, evaluate_bayesian
from utils import save_checkpoint, load_checkpoint, plot_history, plot_accuracy

from torch.utils.data import DataLoader, random_split
from torch.utils.tensorboard import SummaryWriter
from torchvision.utils import make_grid, save_image
from tqdm import tqdm

In [ ]:
'''Data preprocessing for Cifar10'''
# 5 disjoint class pairs covering all 10 CIFAR-10 labels
CLASS_PAIRS = [(0, 1), (2, 3), (4, 5), (6, 7), (8, 9)]

CIFAR10_CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                    'dog', 'frog', 'horse', 'ship', 'truck']

def load_cifar10(root, train=True, download=True, transform=True):
    """Load full CIFAR10 and return all images as a single stacked tensor
    plus a tensor of integer labels."""

    mean = (0.4914, 0.4822, 0.4465)
    std = (0.2470, 0.2435, 0.2616)
    if transform is True:
        transform = transforms.Compose([
            transforms.Pad(4),
            transforms.RandomCrop(32),
            transforms.Resize(32),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])
    elif transform is False:
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])

    dataset = torchvision.datasets.CIFAR10(
        root=root, train=train, download=download, transform=transform
    )

    images = torch.stack([dataset[i][0] for i in range(len(dataset))])
    labels = torch.tensor(dataset.targets)
    return images, labels


def make_binary_group(images, labels, class_pair):
    """Filter images/labels down to the two classes in class_pair and
    remap labels to {0, 1}. Returns a TensorDataset."""
    c0, c1 = class_pair
    mask = (labels == c0) | (labels == c1)

    group_images = images[mask]
    group_labels = labels[mask]

    binary_labels = torch.where(group_labels == c0,
                                 torch.zeros_like(group_labels),
                                 torch.ones_like(group_labels)).float()

    return TensorDataset(group_images, binary_labels)


def build_binary_groups(images, labels, class_pairs=CLASS_PAIRS):
    """Build one TensorDataset per class pair. Returns a list of datasets
    in the same order as class_pairs."""
    return [make_binary_group(images, labels, pair) for pair in class_pairs]


# Load CIFAR10 and build binary datasets for each class pair
train_images, train_labels = load_cifar10(root='../data/cifar10', train=True, transform=True)
test_images, test_labels = load_cifar10(root='../data/cifar10', train=False, transform=False)

train_groups = build_binary_groups(train_images, train_labels, CLASS_PAIRS)
test_groups = build_binary_groups(test_images, test_labels, CLASS_PAIRS)

for pair, tr_g, te_g in zip(CLASS_PAIRS, train_groups, test_groups):
    c0, c1 = CIFAR10_CLASSES[pair[0]], CIFAR10_CLASSES[pair[1]]
    print(f"Pair {pair} ({c0} vs {c1}): train={len(tr_g)}, test={len(te_g)}")

loader = DataLoader(train_groups[0], batch_size=64, shuffle=True)
imgs, labs = next(iter(loader))
print(imgs.shape, labs[:10])


Pair (0, 1) (airplane vs automobile): train=10000, test=2000
Pair (2, 3) (bird vs cat): train=10000, test=2000
Pair (4, 5) (deer vs dog): train=10000, test=2000
Pair (6, 7) (frog vs horse): train=10000, test=2000
Pair (8, 9) (ship vs truck): train=10000, test=2000
torch.Size([64, 3, 32, 32]) tensor([1., 1., 1., 0., 0., 1., 1., 0., 0., 0.])


In [ ]:
# Paths
results_path = '../results/resnet32_bayesian/cifar10_binary'
os.makedirs(results_path, exist_ok=True)

checkpoint_path = os.path.join(results_path, f'best_model_resnet32_bayesian_cifar10_binary.pth')

log_dir = '../logs'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Train model on the first class pair (airplane vs automobile)
train_loader = DataLoader(train_groups[0], batch_size=64, shuffle=True, num_workers=4)
test_loader = DataLoader(test_groups[0], batch_size=64, shuffle=False, num_workers=4)

# HP configurations
lr = 0.001
batch_size = 64
momentum = 0.9
weight_decay = 5e-4
num_epochs = 300
num_mc = 20 # for validation and uncertainty estimation

# Initialze model and optimizer
model = ResNet(n=5, shortcuts=True).to(device)
print(f"Using ResNet-32_bayesian")

optimizer = torch.optim.Adam(model.parameters(), lr)

# Loss function
criterion = nn.CrossEntropyLoss()

logger_dir = os.path.join(log_dir, f"resnet32_bayesian_cifar10_binary")
tb_writer = SummaryWriter(logger_dir)

In [ ]:
# Support function
@torch.no_grad()
def get_mean_sigma(model):
    sigmas = []
    for _, module in model.named_modules():
        rho = getattr(module, "rho_kernel", None)
        if rho is None:
            rho = getattr(module, "rho_weight", None)
        if rho is None:
            continue
        sigmas.append(torch.log1p(torch.exp(rho.detach()) + 1e-12).flatten())
    return torch.cat(sigmas).mean().item() if sigmas else 0.0

@torch.no_grad()
def log_layer_uncertainty(model, writer, epoch):
    for name, module in model.named_modules():
        mu = getattr(module, "mu_kernel", None)
        rho = getattr(module, "rho_kernel", None)

        if mu is None or rho is None:
            mu = getattr(module, "mu_weight", None)
            rho = getattr(module, "rho_weight", None)

        if mu is None or rho is None:
            continue

        mu = mu.detach()
        sigma = torch.log1p(torch.exp(rho.detach()))
        snr = mu.abs() / (sigma + 1e-12)

        writer.add_histogram(f"{name}/mu", mu, epoch)
        writer.add_histogram(f"{name}/sigma", sigma, epoch)
        writer.add_histogram(f"{name}/snr", snr, epoch)
        writer.add_scalar(f"{name}/mean_sigma", sigma.mean().item(), epoch)
        writer.add_scalar(f"{name}/mean_snr", snr.mean().item(), epoch)


In [ ]:

# Training/Validating Loop
print(f"Starting training — {num_epochs} epochs\n")

scaler = torch.cuda.amp.GradScaler()

for epoch in range(0, num_epochs):

    train_acc, train_loss = train_one_epoch_bayesian(
        model, train_loader, num_mc, criterion, optimizer, epoch, device, scaler, tb_writer
    )
    test_acc, test_loss = validate_bayesian(
        model, test_loader, num_mc, criterion, epoch, device, tb_writer
    )

    if (epoch >= 80 and epoch < 120):
        lr = 0.1 * lr
    elif (epoch >= 120 and epoch < 160):
        lr = 0.01 * lr
    elif (epoch >= 160 and epoch < 180):
        lr = 0.001 * lr
    elif (epoch >= 180):
        lr = 0.0005 * lr

    print(
        f"Epoch [{epoch:3d}/{num_epochs}] | LR: {lr:.4f} | "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
        f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.2f}%"
    )

    # per-layer weight uncertainty logging
    log_layer_uncertainty(model, tb_writer, epoch)


    # Save best checkpoint
    if test_acc > best_test_acc:
        best_test_acc = test_acc
        save_checkpoint(
            state={
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'test_acc': test_acc,
            },
            save_path=checkpoint_path
        )

print(f"\nTraining complete.")
print(f"Best test accuracy : {best_test_acc:.2f}%")
print(f"Best checkpoint    : {checkpoint_path}\n")